# Сравнение метрик жестикуляции

Это шаблонный ноутбук для сравнения нескольких способов оценки жестикуляции на заранее собранной выборке видео.

Ноутбук подготовлен как каркас. Сюда нужно будет добавить реальную логику извлечения признаков, прогнать её на настоящих роликах, а перед коммитом очистить outputs.

## Цели сравнения

Обязательно нужно сравнить:
1. Текущий baseline из Charisma Master.
2. Простой нормализованный метод, предложенный нейросетью.
3. Более сложный, но CPU-friendly метод, предложенный нейросетью.

Что должно получиться по итогам:
- хорошие профессиональные ролики должны уверенно отделяться от плохих по сырым метрикам;
- собственные ролики должны оказываться ближе к хорошим, чем к плохим;
- метрика не должна слишком сильно зависеть от расстояния до камеры;
- должно быть измерено время обработки как по каждому ролику, так и по всей выборке.

In [ ]:
from pathlib import Path
from time import perf_counter
from typing import Any

import math

import numpy as np
import pandas as pd

try:
    import cv2
except ImportError:
    cv2 = None

try:
    import mediapipe as mp
except ImportError:
    mp = None


In [ ]:
NOTEBOOK_PATH = Path.cwd()
PROJECT_ROOT = NOTEBOOK_PATH.parent if NOTEBOOK_PATH.name == 'notebooks' else NOTEBOOK_PATH
GESTURE_DIR = PROJECT_ROOT
MEDIA_DIR = GESTURE_DIR / 'media'
RESULTS_DIR = GESTURE_DIR / 'results'
MANIFEST_PATH = GESTURE_DIR / 'dataset_manifest.csv'
COMPARISON_TABLE_PATH = GESTURE_DIR / 'comparison_table.csv'
CONFIG_PATH = GESTURE_DIR / 'config.json'

print('Папка gesture:', GESTURE_DIR)
print('Путь к manifest:', MANIFEST_PATH)
print('Папка с видео:', MEDIA_DIR)

## Загрузка manifest-файла

Ожидаются группы:
- `good_professional`
- `bad`
- `self_recorded`

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df

In [ ]:
required_columns = [
    'video_id',
    'имя_файла',
    'группа',
    'тип_источника',
    'длительность_сек',
    'исходный_fps',
    'исходное_разрешение',
    'категория_дистанции',
    'есть_показывание_на_слайды',
    'заметки',
]
missing_columns = [c for c in required_columns if c not in manifest_df.columns]
assert not missing_columns, f'Отсутствуют колонки в manifest: {missing_columns}'

manifest_df['video_path'] = manifest_df['имя_файла'].map(lambda name: str(MEDIA_DIR / name))
manifest_df[['video_id', 'группа', 'категория_дистанции', 'video_path']]

## Метод 1 — текущий baseline из Charisma Master

Референс в этом репозитории:
- `charisma-master/services/ml_worker/app/logic/ml_engine.py`
- `charisma-master/services/ml_worker/app/config.json`
- `charisma-master/services/ml_worker/app/config_fields.md`

Краткая логика текущего подхода:
- по кадрам накапливается движение при наличии распознанной позы,
- считается среднее движение,
- затем оно масштабируется через `gesture_scale`,
- результат ограничивается сверху `max_score`,
- дальше используются нижний и верхний пороги для интерпретации.

In [ ]:
CHARISMA_BASELINE_CONFIG = {
    'target_frame_width': 480,
    'frame_sample_every': 5,
    'min_frames_with_pose': 10,
    'gesture_scale': 3500,
    'gesture_low_lt': 15,
    'gesture_high_gt': 85,
    'max_score': 100,
}
CHARISMA_BASELINE_CONFIG

In [ ]:
def run_charisma_master_baseline(video_path: str, config: dict[str, Any]) -> dict[str, Any]:
    """
    TODO: перенести или обернуть сюда текущую baseline-реализацию после переноса в целевой проект.

    Ожидаемый формат результата:
    {
        'raw_metric': float,
        'score': float,
        'frames_processed': int,
        'frames_with_pose': int,
        'processing_time_sec': float,
    }
    """
    start = perf_counter()
    # TODO: реализовать через MediaPipe / логику, перенесённую из Charisma Master
    result = {
        'raw_metric': np.nan,
        'score': np.nan,
        'frames_processed': 0,
        'frames_with_pose': 0,
        'processing_time_sec': perf_counter() - start,
    }
    return result


## Метод 2 — простой нормализованный метод

Возможные базовые признаки:
- нормализованная скорость движения рук относительно ширины плеч,
- амплитуда жестов относительно масштаба корпуса,
- доля кадров с активной жестикуляцией,
- при необходимости — доля кадров с жестами, похожими на показывание на слайды.

Цель: добиться лучшего разделения, чем у baseline, но сохранить простоту и скорость.

In [ ]:
def run_simple_normalized_method(video_path: str) -> dict[str, Any]:
    """
    TODO: реализовать простой нормализованный способ оценки жестикуляции.

    Возможные компоненты:
    - скорость рук, нормализованная по ширине плеч
    - отклонение рук от нейтрального положения
    - доля кадров с активным движением
    - количество или доля pointing-подобных жестов
    """
    start = perf_counter()
    result = {
        'raw_metric': np.nan,
        'score': np.nan,
        'activity_ratio': np.nan,
        'amplitude_mean': np.nan,
        'pointing_ratio': np.nan,
        'processing_time_sec': perf_counter() - start,
    }
    return result


## Метод 3 — более сложный, но CPU-friendly метод

Что можно добавить по сравнению с простым методом:
- сегментацию на эпизоды жестикуляции,
- анализ ритма и структуры пауз,
- анализ вариативности и повторяемости движений,
- штрафы за чрезмерную статичность или чрезмерную хаотичность.

Цель: получить более содержательный сигнал, оставаясь в пределах нормальной работы на CPU.

In [ ]:
def run_cpu_structured_method(video_path: str) -> dict[str, Any]:
    """
    TODO: реализовать более богатую временную метрику жестикуляции.

    Возможные компоненты:
    - выделение эпизодов жестикуляции
    - частота эпизодов и их средняя длительность
    - регулярность ритма
    - вариативность движений
    - штрафы за зажатость или хаотичность
    """
    start = perf_counter()
    result = {
        'raw_metric': np.nan,
        'score': np.nan,
        'episode_count': np.nan,
        'episode_duration_mean': np.nan,
        'rhythm_score': np.nan,
        'processing_time_sec': perf_counter() - start,
    }
    return result


## Пакетный запуск

Ниже — каркас для запуска всех методов на всех роликах с сохранением значений метрик и времени обработки.

In [ ]:
METHODS = {
    'charisma_master_baseline': lambda video_path: run_charisma_master_baseline(video_path, CHARISMA_BASELINE_CONFIG),
    'llm_simple_normalized': run_simple_normalized_method,
    'llm_cpu_structured': run_cpu_structured_method,
}

def run_all_methods(manifest: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for item in manifest.to_dict(orient='records'):
        for method_name, method_fn in METHODS.items():
            metrics = method_fn(item['video_path'])
            rows.append({
                'video_id': item['video_id'],
                'имя_файла': item['имя_файла'],
                'группа': item['группа'],
                'категория_дистанции': item['категория_дистанции'],
                'метод': method_name,
                **metrics,
            })
    return pd.DataFrame(rows)

# results_df = run_all_methods(manifest_df)
# results_df.head()

## Стандартизированный разрыв

Главная обязательная метрика сравнения:
- стандартизированный разрыв между `good_professional` и `bad` по сырой метрике.

В качестве базового варианта можно использовать аналог effect size с объединённым стандартным отклонением.

In [ ]:
def standardized_gap(good_values: pd.Series, bad_values: pd.Series) -> float:
    good = pd.to_numeric(good_values, errors='coerce').dropna()
    bad = pd.to_numeric(bad_values, errors='coerce').dropna()
    if len(good) < 2 or len(bad) < 2:
        return float('nan')

    good_mean = good.mean()
    bad_mean = bad.mean()
    good_var = good.var(ddof=1)
    bad_var = bad.var(ddof=1)
    pooled_std = math.sqrt(((len(good) - 1) * good_var + (len(bad) - 1) * bad_var) / (len(good) + len(bad) - 2))
    if pooled_std == 0:
        return float('nan')
    return float((good_mean - bad_mean) / pooled_std)


## Построение итоговой сравнительной таблицы

Ожидаемые колонки:
- метод
- описание метода
- стандартизированный разрыв good vs bad
- общее время обработки
- время обработки по роликам
- при желании — плюсы и минусы

In [ ]:
METHOD_DESCRIPTIONS = {
    'charisma_master_baseline': 'Текущий baseline Charisma Master на основе среднего движения рук, масштабированного в score.',
    'llm_simple_normalized': 'Простой нормализованный метод на основе активности жестикуляции, амплитуды и масштабно-нормализованного движения.',
    'llm_cpu_structured': 'Более сложный CPU-friendly метод на основе эпизодов жестикуляции, ритма и вариативности движений.',
}

def build_comparison_table(results_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method_name, method_df in results_df.groupby('метод'):
        good_values = method_df.loc[method_df['группа'] == 'good_professional', 'raw_metric']
        bad_values = method_df.loc[method_df['группа'] == 'bad', 'raw_metric']
        gap = standardized_gap(good_values, bad_values)
        total_time = pd.to_numeric(method_df['processing_time_sec'], errors='coerce').sum()
        per_video_times = pd.to_numeric(method_df['processing_time_sec'], errors='coerce').round(3).tolist()
        rows.append({
            'метод': method_name,
            'описание_метода': METHOD_DESCRIPTIONS.get(method_name, ''),
            'стандартизированный_разрыв_good_vs_bad': gap,
            'общее_время_обработки_сек': total_time,
            'время_обработки_по_роликам_сек': per_video_times,
            'плюсы': '',
            'минусы': '',
        })
    return pd.DataFrame(rows).sort_values('стандартизированный_разрыв_good_vs_bad', ascending=False)

# comparison_df = build_comparison_table(results_df)
# comparison_df

## Анализ чувствительности к расстоянию

Здесь нужно проверить, насколько стабильно ведёт себя метрика на `self_recorded` роликах, записанных на `near`, `mid` и `far`, если манера речи и жестикуляции остаётся примерно одинаковой.

In [ ]:
def analyze_distance_sensitivity(results_df: pd.DataFrame) -> pd.DataFrame:
    self_df = results_df[results_df['группа'] == 'self_recorded'].copy()
    summary_rows = []
    for method_name, method_df in self_df.groupby('метод'):
        pivot = method_df.pivot_table(index='video_id', columns='категория_дистанции', values='raw_metric', aggfunc='first')
        near_mid_far_spread = float('nan')
        if not pivot.empty:
            near_mid_far_spread = float((pivot.max(axis=1) - pivot.min(axis=1)).mean())
        summary_rows.append({
            'метод': method_name,
            'средний_разброс_на_собственных_роликах': near_mid_far_spread,
        })
    return pd.DataFrame(summary_rows).sort_values('средний_разброс_на_собственных_роликах')

# distance_df = analyze_distance_sensitivity(results_df)
# distance_df

## Выбор алгоритма и подбор порогов

Когда появятся реальные результаты, финальный метод нужно выбирать по совокупности факторов:
- насколько велик и полезен стандартизированный разрыв между good и bad;
- приемлемо ли время работы на CPU;
- хорошо ли ведут себя собственные ролики;
- не слишком ли чувствительна метрика к расстоянию до камеры.

После этого результаты нужно обсудить с тимлидом и заполнить `config.json` и `config.md`.

In [ ]:
def export_results(results_df: pd.DataFrame, comparison_df: pd.DataFrame) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(RESULTS_DIR / 'per_video_metrics.csv', index=False)
    comparison_df.to_csv(COMPARISON_TABLE_PATH, index=False)

# export_results(results_df, comparison_df)

## Финальный чек-лист

Перед завершением работы в целевом проекте нужно проверить:
- [ ] Все реальные ролики добавлены в `media/`.
- [ ] `dataset_manifest.csv` заполнен.
- [ ] Все три метода реализованы.
- [ ] Сравнение запущено на всей выборке.
- [ ] `comparison_table.csv` заполнен.
- [ ] Лучший алгоритм и пороги обсуждены с тимлидом.
- [ ] `config.json` заполнен.
- [ ] Обоснование занесено в `config.md`.
- [ ] Outputs ноутбука очищены перед коммитом.